In [0]:
SOURCE_PATH = (
    "/Volumes/workspace/default/"
    "ubuntu_dialogue_raw/dialogueText_196.csv"
)

uploaded_files = dbutils.fs.ls(
    "/Volumes/workspace/default/ubuntu_dialogue_raw"
)

[(item.name, item.size) for item in uploaded_files]

In [0]:
bronze_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .csv(SOURCE_PATH)
)

print("Columns:", bronze_df.columns)
display(bronze_df.limit(10))

In [0]:
from time import perf_counter

BRONZE_TABLE = "workspace.default.bronze_ubuntu_dialogue"

started = perf_counter()

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

bronze_ingestion_seconds = perf_counter() - started

stored_bronze_df = spark.table(BRONZE_TABLE)
bronze_row_count = stored_bronze_df.count()

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Rows: {bronze_row_count:,}")
print(f"Ingestion time: {bronze_ingestion_seconds:,.2f} seconds")

In [0]:
display(
    spark.sql(
        "DESCRIBE DETAIL workspace.default.bronze_ubuntu_dialogue"
    )
)

In [0]:
%pip install \
  pyspellchecker==0.8.3 \
  vaderSentiment==3.3.2 \
  tqdm==4.67.1

In [0]:
%restart_python

In [0]:
import os
import sys

REPO_ROOT = (
    "/Workspace/Users/"
    "lwool@sandiego.edu/"
    "Ubuntu-dialogue-corpus"
)

if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f"Git folder not found: {REPO_ROOT}")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from pipeline.pipeline import PipelineConfig
from databricks_integration.scripts.bronze_to_silver_ubuntu import (
    append_silver_audit_delta,
    run_bronze_to_silver_spark,
    write_silver_delta,
)

print("Project imports succeeded.")
print("Repository:", REPO_ROOT)

In [0]:
from databricks_integration.scripts.bronze_to_silver_ubuntu import (
    append_silver_audit_delta,
    release_silver_resources,
    run_bronze_to_silver_spark,
    write_silver_delta,
)

In [0]:
BRONZE_TABLE = "workspace.default.bronze_ubuntu_dialogue"

smoke_config = PipelineConfig(
    residual_policy="manual_review",
    sentiment_mode="vader",
    advanced_nlp=False,
    sample_size=1_000,
    sample_seed=42,
)

smoke_silver_df = None
smoke_metadata = None

try:
    smoke_silver_df, smoke_validation, smoke_metadata = (
        run_bronze_to_silver_spark(
            spark,
            spark.table(BRONZE_TABLE),
            config=smoke_config,
            materialization_mode="delta",
            materialization_schema="workspace.default",
        )
    )

    print("Validation:", smoke_validation)
    print("Materialization:", smoke_metadata["materialization"])

    display(
        smoke_silver_df.select(
            "message_id",
            "text",
            "text_cleaned",
            "vader_compound",
            "vader_label",
        ).limit(10)
    )

finally:
    if smoke_silver_df is not None and smoke_metadata is not None:
        release_silver_resources(
            spark,
            smoke_silver_df,
            smoke_metadata,
        )

In [0]:
#Same as cell 9, except testing my overlay rather than manual review
BRONZE_TABLE = "workspace.default.bronze_ubuntu_dialogue"

smoke_config = PipelineConfig(
    residual_policy="reviewed", #ONLY CHANGE TO CELL 9
    sentiment_mode="vader",
    advanced_nlp=False,
    sample_size=1_000,
    sample_seed=42,
)

smoke_silver_df = None
smoke_metadata = None

try:
    smoke_silver_df, smoke_validation, smoke_metadata = (
        run_bronze_to_silver_spark(
            spark,
            spark.table(BRONZE_TABLE),
            config=smoke_config,
            materialization_mode="delta",
            materialization_schema="workspace.default",
        )
    )

    print("Validation:", smoke_validation)
    print("Materialization:", smoke_metadata["materialization"])

    display(
        smoke_silver_df.select(
            "message_id",
            "text",
            "text_cleaned",
            "vader_compound",
            "vader_label",
        ).limit(10)
    )

finally:
    if smoke_silver_df is not None and smoke_metadata is not None:
        release_silver_resources(
            spark,
            smoke_silver_df,
            smoke_metadata,
        )

Smoke test complete! Now, time to shift into a Data Engineering job.